# CouchMo — MetaDrive RL training (Colab)

Thin orchestration notebook. See `training/metadrive/README.md` for context.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%%bash
cd /content
if [ ! -d CouchMo ]; then
  git clone https://github.com/<USER>/CouchMo.git
else
  cd CouchMo && git pull
fi

In [ ]:
%cd /content/CouchMo

In [ ]:
!pip install -q -r training/requirements.txt -r training/requirements-rl.txt

In [ ]:
# Smoke test: env reset/step on a single scenario.
!python -c "from training.metadrive.env import CouchMoMetaDriveEnv; import numpy as np\nenv = CouchMoMetaDriveEnv(config={'num_scenarios': 5}); obs, _ = env.reset(seed=0); print('obs', obs.shape, obs.dtype); env.step(np.array([0.0, 0.3], dtype=np.float32)); env.close(); print('OK')"

In [ ]:
# BC data collection — skip if Drive already has the dataset.
import pathlib
DATA = '/content/drive/MyDrive/CouchMo/metadrive_bc'
if not (pathlib.Path(DATA) / 'manifest.json').exists():
    !python -m training.metadrive.collect_bc --data-root {DATA} --episodes 500 --max-steps 500
else:
    print('BC dataset already on Drive at', DATA)

In [ ]:
# BC training.
!python -m training.imitation.train_bc --data-root /content/drive/MyDrive/CouchMo/metadrive_bc --out /content/drive/MyDrive/CouchMo/checkpoints/bc_metadrive.pt --epochs 10

In [ ]:
# PPO fine-tune. Auto-resumes if checkpoints exist under --output-dir.
!python -m training.metadrive.train_ppo --bc-ckpt /content/drive/MyDrive/CouchMo/checkpoints/bc_metadrive.pt --output-dir /content/drive/MyDrive/CouchMo/checkpoints/ppo_runs/run_001 --total-timesteps 5000000 --n-envs 8

In [ ]:
%load_ext tensorboard
%tensorboard --logdir /content/drive/MyDrive/CouchMo/checkpoints/ppo_runs/run_001/tb

In [ ]:
# Export best checkpoint to ONNX.
!python -m training.imitation.export_onnx --from-ppo --in /content/drive/MyDrive/CouchMo/checkpoints/ppo_runs/run_001/best_model.zip --out /content/drive/MyDrive/CouchMo/exports/couchmo_v1.onnx --verify

In [ ]:
# Go/no-go gate. Non-zero exit = not ready for real hardware.
!python -m training.metadrive.eval_policy --onnx /content/drive/MyDrive/CouchMo/exports/couchmo_v1.onnx --episodes 500